# 08 · Quantum Frequency Combs

**Covers:** microring resonances, spontaneous four-wave mixing as two-mode squeezing, signal–idler correlations, the rate vs purity trade-off, frequency-bin entanglement and multi-pair events.
**Photonics thread:** this is the research direction of Kues et al. ([Nature 546, 622, 2017](https://iris.unipa.it/handle/10447/238845)): one integrated microring producing photon pairs entangled across many frequency bins, controlled with standard telecom components.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

## 1. A ring is a ladder of modes

**Plain words:** only colours that fit a whole number of wavelengths around the loop survive, and they are equally spaced by the free spectral range:

$$\mathrm{FSR} = \frac{c}{n_g L},\qquad \Delta\lambda = \frac{\lambda^2\,\mathrm{FSR}}{c}$$

Each resonance is a mode (a "bucket" from notebook 02). Pumped hard, the lines form a classical comb; pumped gently, they carry correlated photon pairs.

**Gap:** equal spacing needs dispersion engineering, and it is never perfect on a real chip.

In [2]:
c, lam, fsr = 2.998e8, 1550e-9, 200e9
check("200 GHz FSR at 1550 nm in nm", lam**2 * fsr / c * 1e9, 1.603)
n_g = 2.4                                                    # illustrative group index
L = c / (n_g * fsr)
print(f"ring length {L * 1e3:.3f} mm, radius {L / (2 * np.pi) * 1e6:.0f} um (for n_g = {n_g})")

PASS  200 GHz FSR at 1550 nm in nm: simulation = 1.6027, theory = 1.6030
ring length 0.625 mm, radius 99 um (for n_g = 2.4)


## 2. Four-wave mixing makes pairs (two-mode squeezing)

Two pump photons become a signal and an idler, symmetric around the pump: $2\omega_p = \omega_s + \omega_i$. If the signal is 3 lines above the pump, the idler is 3 lines below. The state is two-mode squeezed vacuum:

$$|\psi\rangle = \frac{1}{\cosh r}\sum_n \tanh^n r\,|n_s, n_i\rangle,\qquad \mu = \sinh^2 r\ \text{photons per arm}$$

- Signal and idler counts are always equal: $\mathrm{Var}(n_s - n_i) = 0$.
- One arm alone is **thermal**: $g^{(2)} = 2$ (tracing out the partner leaves a mixed state).
- Cross-correlation $g_{si} = 2 + 1/\mu$, related to the coincidence-to-accidental ratio (CAR). Conventions differ by ±1 between papers.

**Gap:** in this ideal model $g_{si}$ survives loss; dark counts and Raman noise do not, so measured CAR is lower.

**One sentence:** a gently pumped ring makes signal–idler pairs, and pumping harder trades purity for rate.

In [3]:
N = 20
a_s = qt.tensor(qt.destroy(N), qt.qeye(N))
a_i = qt.tensor(qt.qeye(N), qt.destroy(N))
n_s, n_i = a_s.dag() * a_s, a_i.dag() * a_i
vac = qt.tensor(qt.basis(N, 0), qt.basis(N, 0))

def ket(ns, ni):
    return qt.tensor(qt.basis(N, ns), qt.basis(N, ni))

def pair_state(r):
    return (r * (a_s.dag() * a_i.dag() - a_s * a_i)).expm() * vac

r = 0.2
psi = pair_state(r)
for k in range(3):
    check(f"P({k},{k})", abs(ket(k, k).dag() * psi) ** 2, np.tanh(r) ** (2 * k) / np.cosh(r) ** 2)
check("P(1,0): never a photon alone", abs(ket(1, 0).dag() * psi) ** 2, 0.0)
check("Var(n_s - n_i)", qt.variance(n_s - n_i, psi), 0.0)
mu = qt.expect(n_s, psi)
check("mu = sinh^2 r", mu, np.sinh(r) ** 2)
check("g2 of one arm (thermal)", qt.expect(a_s.dag() ** 2 * a_s ** 2, psi) / mu**2, 2.0)
for rr in [0.1, 0.2, 0.4]:
    p = pair_state(rr)
    m = qt.expect(n_s, p)
    check(f"g_si = 2 + 1/mu at r = {rr}", qt.expect(n_s * n_i, p) / m**2, 2 + 1 / m)

rho_s = qt.ket2dm(psi).ptrace(0)
nb_ = np.sinh(r) ** 2
check("entropy of one arm (bits)", qt.entropy_vn(rho_s, base=2),
      (nb_ + 1) * np.log2(nb_ + 1) - nb_ * np.log2(nb_))

PASS  P(0,0): simulation = 0.9610, theory = 0.9610
PASS  P(1,1): simulation = 0.0374, theory = 0.0374
PASS  P(2,2): simulation = 0.0015, theory = 0.0015
PASS  P(1,0): never a photon alone: simulation = 0.0000, theory = 0.0000
PASS  Var(n_s - n_i): simulation = 0.0000, theory = 0.0000
PASS  mu = sinh^2 r: simulation = 0.0405, theory = 0.0405
PASS  g2 of one arm (thermal): simulation = 2.0000, theory = 2.0000
PASS  g_si = 2 + 1/mu at r = 0.1: simulation = 101.6673, theory = 101.6673
PASS  g_si = 2 + 1/mu at r = 0.2: simulation = 26.6693, theory = 26.6693


PASS  g_si = 2 + 1/mu at r = 0.4: simulation = 7.9271, theory = 7.9271
PASS  entropy of one arm (bits): simulation = 0.2471, theory = 0.2471


## 3. Frequency-bin entanglement

**Plain words:** the ring has many matched line pairs (+1/−1, +2/−2, ...). Weakly pumped, one pair is born in all of them at once, with nothing revealing which. With $d$ line pairs:

$$|\Phi\rangle = \frac{1}{\sqrt d}\sum_k |k\rangle_s|k\rangle_i$$

This is a Bell state with $d$ levels instead of 2 (a qudit pair), carrying $\log_2 d$ ebits.

**Proof it isn't classical:** measure both photons in the **Fourier basis** (the frequency analogue of 45° polarization). The entangled state stays perfectly correlated ($m + m' = 0 \bmod d$, probability $1/d$ each); a classical "same line" mixture gives every outcome probability $1/d^2$. In the lab this mixing of frequency bins uses electro-optic modulators and pulse shapers.

**Gap:** real comb lines have unequal amplitudes, so a real state is a weighted version of this one.

**One sentence:** one ring makes a pair entangled across many frequency bins, and the Fourier-basis test proves it.

In [4]:
d = 3
w = np.exp(2j * np.pi / d)
Phi = sum(qt.tensor(qt.basis(d, k), qt.basis(d, k)) for k in range(d)).unit()
rho1 = qt.ket2dm(Phi).ptrace(0)
check("purity of one photon", (rho1 * rho1).tr(), 1 / d)
check("entanglement (bits)", qt.entropy_vn(rho1, base=2), np.log2(d))

def fourier(m):
    return qt.Qobj(np.array([[w ** (m * k)] for k in range(d)]) / np.sqrt(d))

rho_ent = qt.ket2dm(Phi)
rho_cl = sum(qt.ket2dm(qt.tensor(qt.basis(d, k), qt.basis(d, k))) for k in range(d)) / d
M_ent = np.zeros((d, d)); M_cl = np.zeros((d, d))
for m in range(d):
    for m2 in range(d):
        proj = qt.ket2dm(qt.tensor(fourier(m), fourier(m2)))
        M_ent[m, m2] = qt.expect(proj, rho_ent).real
        M_cl[m, m2] = qt.expect(proj, rho_cl).real
print("entangled:\n", np.abs(M_ent).round(3))
print("classical:\n", np.abs(M_cl).round(3))
check("entangled: P(m=1, m'=2)", M_ent[1, 2], 1 / d)
check("entangled: P(m=1, m'=1)", M_ent[1, 1], 0.0)
check("classical: every outcome", M_cl[1, 1], 1 / d**2)

PASS  purity of one photon: simulation = 0.3333, theory = 0.3333
PASS  entanglement (bits): simulation = 1.5850, theory = 1.5850
entangled:
 [[0.333 0.    0.   ]
 [0.    0.    0.333]
 [0.    0.333 0.   ]]
classical:
 [[0.111 0.111 0.111]
 [0.111 0.111 0.111]
 [0.111 0.111 0.111]]
PASS  entangled: P(m=1, m'=2): simulation = 0.3333, theory = 0.3333
PASS  entangled: P(m=1, m'=1): simulation = 0.0000, theory = 0.0000
PASS  classical: every outcome: simulation = 0.1111, theory = 0.1111


## 4. The multi-pair problem

Each line pair is an independent source with $x = \tanh^2 r$. Across $d$ line pairs the chance of exactly $n$ pairs in total is $\binom{n+d-1}{d-1}x^n(1-x)^d$. Gentle pumping keeps the state clean but makes pairs rare.

In [5]:
from math import comb
x = np.tanh(r) ** 2
def p_total(k, lines=3):
    return comb(k + lines - 1, lines - 1) * x**k * (1 - x) ** lines
p0, p1 = p_total(0), p_total(1)
print(f"no pair {p0:.3f}, one pair {p1:.3f}, two or more {1 - p0 - p1:.3f}")
check("fraction of pair events that are multi-pair", (1 - p0 - p1) / (1 - p0), 0.077)

no pair 0.888, one pair 0.104, two or more 0.009
PASS  fraction of pair events that are multi-pair: simulation = 0.0769, theory = 0.0770


## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [6]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

18 of 18 checks passed
